# Phase 3.4 (Notebook A): Pipeline — StandardScaler ML export

This notebook is **Notebook A (StandardScaler)**. It writes **`phase2_df_undersample_*_ml.parquet`** under `BASE_GROUP` for **LR, MLP, RFC, LSVC** when training on Phase 3.3 **undersampled** train (`df_train_phase3_downsampled.parquet`).

**Sibling:** [Phase2.4_Pipeline_AssemblerOnly_ML.ipynb](Phase2.4_Pipeline_AssemblerOnly_ML.ipynb) (Notebook B) exports assembler-only features for **GBT** (`phase2_df_rawasm_undersample_*_ml.parquet`).

| Pipeline | Parquet prefix | Phase 2.5 models |
|----------|----------------|------------------|
| **A (this notebook)** | `phase2_df_undersample_*_ml` | LR, MLP, RFC, LSVC (undersampled train) |
| **B** | `phase2_df_rawasm_undersample_*_ml` | GBT |

Prerequisite: Phase 3.3 `df_*_phase3.parquet` and `phase3_final_num_cols.json`. **Re-run** after any Phase 3.3 schema change.

**Phase 3.3 FE alignment:** `avg_Hourly*_scl` use **rolling median + MAD** scaling; `dep_hour_sin2` / `dep_hour_cos2` appear when present in `phase3_final_num_cols.json`. **Train balance** is fixed in Phase 3.3 only; Phase 2.5 LR/RF/GBT notebooks **do not** resample the train majority again.

**Sync note:** When changing OOM or assemble logic, apply the same edit to Notebook B (only the config cell differs).
**OOM mitigation:** narrow projection before `cache()`/`fit`, tunable `SPARK_SHUFFLE_PARTITIONS`, optional `TRAIN_REPARTITION_BEFORE_FIT`. If executor OOM persists, ask an admin for larger workers.


In [0]:
import json
import pyspark.sql.functions as F
from pyspark.sql.types import *
import pandas as pd
import numpy as np

from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import StandardScaler, VectorAssembler

print("Libraries loaded")

Libraries loaded


## Load data

Phase 3.3 (undersample path) writes **`df_train_phase3_downsampled.parquet`** (train), **`df_val_phase3.parquet`**, **`df_test_phase3.parquet`**, and **`phase3_final_num_cols.json`** under `BASE_GROUP`.

**This notebook (A)** writes ML parquets with prefix **`phase2_df_undersample`** (e.g. `phase2_df_undersample_train_ml.parquet`) for **LR, MLP, RFC, LSVC** in [phase2.5_modeling_final.ipynb](phase2.5_modeling_final.ipynb) when using the undersampled pipeline.

Set `USE_PHASE3_COLUMN_LIST = False` only for legacy `df_*_fe.parquet` plus hardcoded column lists.


In [0]:
BASE_GROUP = "dbfs:/student-groups/Group_01_02"
TARGET = "DEP_DEL15"
WRITE_PIPELINE_MODEL = True
WRITE_ML_PARQUETS = True
PIPELINE_PATH = f"{BASE_GROUP}/phase3_pipeline_model_standard_undersample"

USE_ML_WEATHER_IMPUTER = False  # Keep False if parquets come from Phase 3.3 (already 7d rolling imputed). True only to re-run the same 7d logic here.
USE_STANDARD_SCALER = True  # LOCKED (Notebook A): StandardScaler (train-fitted mean/std) for LR / MLP / RFC / LSVC exports.

USE_PHASE3_COLUMN_LIST = True  # False: legacy hardcoded numerical_cols + df_*_fe.parquet paths below
FINAL_NUM_COLS_PATH = f"{BASE_GROUP}/phase3_final_num_cols.json"

# --- OOM mitigation (tunable; no cluster resize required) ---
VERIFY_ROW_COUNTS = True  # False: skip full count() scans (faster, less driver pressure; skips row-equality asserts later)
SPARK_SHUFFLE_PARTITIONS = 1024  # more partitions → smaller tasks; tune 512–2048 to cluster cores
SPARK_MAX_PARTITION_BYTES = "67108864"  # 64 MiB parquet split hint; smaller → more tasks
TRAIN_REPARTITION_BEFORE_FIT = None  # e.g. 256: repartition train only before fit (extra shuffle; helps fat partitions)
DISABLE_MLFLOW_AUTOLOG_FOR_FIT = True  # Databricks: reduce mlflow.patch overhead around Estimator.fit

spark.conf.set("spark.sql.shuffle.partitions", str(SPARK_SHUFFLE_PARTITIONS))
spark.conf.set("spark.sql.files.maxPartitionBytes", SPARK_MAX_PARTITION_BYTES)
print(
    f"Spark: shuffle.partitions={SPARK_SHUFFLE_PARTITIONS}, "
    f"files.maxPartitionBytes={SPARK_MAX_PARTITION_BYTES}"
)

TRAIN_PARQUET = f"{BASE_GROUP}/df_train_phase3_downsampled.parquet"  # undersampled train from Phase 3.3
VAL_PARQUET = f"{BASE_GROUP}/df_val_phase3.parquet"
TEST_PARQUET = f"{BASE_GROUP}/df_test_phase3.parquet"

# Legacy FE parquet paths (used only when USE_PHASE3_COLUMN_LIST is False)
TRAIN_PARQUET_FE = f"{BASE_GROUP}/df_train_fe.parquet"
VAL_PARQUET_FE = f"{BASE_GROUP}/df_val_fe.parquet"
TEST_PARQUET_FE = f"{BASE_GROUP}/df_test_fe.parquet"

# ML export: phase2_df_undersample_* matches phase2.5 undersample pipeline reads
ML_PARQUET_PREFIX = "phase2_df_undersample"  # Notebook A + undersampled train (parallel to phase2_df_*)
ML_TRAIN_ML_PARQUET = f"{BASE_GROUP}/{ML_PARQUET_PREFIX}_train_ml.parquet"
ML_VAL_ML_PARQUET = f"{BASE_GROUP}/{ML_PARQUET_PREFIX}_val_ml.parquet"
ML_TEST_ML_PARQUET = f"{BASE_GROUP}/{ML_PARQUET_PREFIX}_test_ml.parquet"

if USE_PHASE3_COLUMN_LIST:
    raw = dbutils.fs.head(FINAL_NUM_COLS_PATH, max_bytes=10_000_000)
    FINAL_NUM_COLS = json.loads(raw)
    print(f"Loaded phase3_final_num_cols.json: {len(FINAL_NUM_COLS)} names")
    _train_p, _val_p, _test_p = TRAIN_PARQUET, VAL_PARQUET, TEST_PARQUET
else:
    FINAL_NUM_COLS = None
    _train_p, _val_p, _test_p = TRAIN_PARQUET_FE, VAL_PARQUET_FE, TEST_PARQUET_FE

df_train_fe = spark.read.parquet(_train_p)
df_val_fe = spark.read.parquet(_val_p)
df_test_fe = spark.read.parquet(_test_p)

_nc = len(df_train_fe.columns)
print(f"Parquet loaded (wide): {_nc} columns. Narrow projection + cache run after assemble cell.")

if VERIFY_ROW_COUNTS:
    print(f"Train: {df_train_fe.count():,} rows")
    print(f"Val: {df_val_fe.count():,} rows")
    print(f"Test: {df_test_fe.count():,} rows")
else:
    print("VERIFY_ROW_COUNTS=False: skipping full-table row counts on wide load.")

# Intentionally no .cache() here — avoids caching ~100+ unused columns before projection.
_PIPELINE_ROW_COUNTS = None  # set after narrow select in assemble cell when VERIFY_ROW_COUNTS


Spark: shuffle.partitions=1024, files.maxPartitionBytes=67108864
Loaded phase3_final_num_cols.json: 103 names
Parquet loaded (wide): 199 columns. Narrow projection + cache run after assemble cell.
Train: 14,515,569 rows
Val: 6,329,538 rows
Test: 6,241,008 rows


## Check what columns we have

Stale cell outputs below may reflect an **older** parquet; **re-run** from the top after Phase 3.3 writes extended columns.


In [0]:
cols = df_train_fe.columns
S = set(cols)
_sorted = sorted(cols)
_n = len(_sorted)
if _n <= 25:
    print(f"Parquet columns ({_n}): {_sorted}")
else:
    print(f"Parquet columns: {_n} (sample: {_sorted[:15]} ... {_sorted[-5:]})")

if USE_PHASE3_COLUMN_LIST and FINAL_NUM_COLS is not None:
    _miss = [c for c in FINAL_NUM_COLS if c not in S]
    _ok = len(FINAL_NUM_COLS) - len(_miss)
    print(f"phase3_final_num_cols.json in parquet: {_ok} / {len(FINAL_NUM_COLS)}")
    if _miss:
        print(f"  not in parquet: {_miss[:12]}{'...' if len(_miss) > 12 else ''}")

if USE_PHASE3_COLUMN_LIST:
    _required = [TARGET]  # undersampled Phase 3.3 has no class_weight; legacy parquets may still have it
else:
    _required = [
        TARGET,
        "ORIGIN",
        "DEST",
        "DAY_OF_WEEK",
        "carrier_delay_rate",
        "origin_delay_rate",
        "dest_delay_rate",
        "route_delay_rate",
        "hour_delay_rate",
    ]
_missing = [c for c in _required if c not in S]
if _missing:
    raise ValueError(f"Missing expected columns: {_missing}")

if USE_PHASE3_COLUMN_LIST and not S.intersection(
    {"is_holiday", "origin_sched_count_prev2h", "wx_wind_x_precip_scl"}
):
    print("[WARN] No Phase 3.3 sentinel cols — check df_*_phase3 paths or re-run Phase 3.3.")

if not USE_PHASE3_COLUMN_LIST:
    if "OP_UNIQUE_CARRIER" not in cols and "OP_CARRIER" not in cols:
        raise ValueError("Need OP_UNIQUE_CARRIER or OP_CARRIER.")

_avg_base = any(c.startswith("avg_Hourly") and not c.endswith("_scl") for c in cols)
if not _avg_base:
    if USE_PHASE3_COLUMN_LIST:
        print("Note: no avg_Hourly* base columns; continuing.")
    else:
        raise ValueError("No avg_Hourly* base columns; check FE parquet.")

_n_scl = sum(1 for c in cols if c.endswith("_scl"))
print(f"*_scl count: {_n_scl} (these skip StandardScaler when in numerical_cols)")

Parquet columns: 199 (sample: ['CRS_ARR_TIME', 'CRS_DEP_TIME', 'DAY_OF_MONTH', 'DAY_OF_WEEK', 'DEP_DEL15', 'DEST', 'DEST_AIRPORT_ID', 'DEST_AIRPORT_SEQ_ID', 'DEST_CITY_MARKET_ID', 'DEST_CITY_NAME', 'DEST_STATE_ABR', 'DEST_STATE_FIPS', 'DEST_STATE_NM', 'DEST_WAC', 'DISTANCE'] ... ['wx_rh_x_precip', 'wx_temp_scl_x_wind_scl', 'wx_vis_bin_x_rh_scl', 'wx_vis_x_wind', 'wx_wind_x_precip'])
phase3_final_num_cols.json in parquet: 103 / 103
*_scl count: 18 (these skip StandardScaler when in numerical_cols)


In [0]:
for c in sorted(df_train_fe.columns):
    print(c)

CRS_ARR_TIME
CRS_DEP_TIME
DAY_OF_MONTH
DAY_OF_WEEK
DEP_DEL15
DEST
DEST_AIRPORT_ID
DEST_AIRPORT_SEQ_ID
DEST_CITY_MARKET_ID
DEST_CITY_NAME
DEST_STATE_ABR
DEST_STATE_FIPS
DEST_STATE_NM
DEST_WAC
DISTANCE
DISTANCE_GROUP
FL_DATE
MONTH
OP_CARRIER
OP_CARRIER_AIRLINE_ID
OP_CARRIER_FL_NUM
OP_UNIQUE_CARRIER
ORIGIN
ORIGIN_AIRPORT_ID
ORIGIN_AIRPORT_SEQ_ID
ORIGIN_CITY_MARKET_ID
ORIGIN_CITY_NAME
ORIGIN_STATE_ABR
ORIGIN_STATE_FIPS
ORIGIN_STATE_NM
ORIGIN_WAC
QUARTER
ROUTE
TAIL_NUM
YEAR
_sort_date
avg_HourlyAltimeterSetting
avg_HourlyDewPointTemperature
avg_HourlyDryBulbTemperature
avg_HourlyDryBulbTemperature_scl
avg_HourlyPrecipitation
avg_HourlyPrecipitation_scl
avg_HourlyPressureChange
avg_HourlyRelativeHumidity
avg_HourlyRelativeHumidity_scl
avg_HourlySeaLevelPressure
avg_HourlySeaLevelPressure_scl
avg_HourlyStationPressure
avg_HourlyVisibility
avg_HourlyVisibility_scl
avg_HourlyWetBulbTemperature
avg_HourlyWindDirection
avg_HourlyWindDirection_Cos
avg_HourlyWindDirection_Sin
avg_HourlyWindGustSpee

## Build `features` (FE-aligned)

- high-cardinality airports/routes are **target-encoded** in Phase 3.3 (`*_delay_rate`).
- **`_scl` pass-through:** Phase 3.3 applies rolling median + MAD scaling into `avg_Hourly*_scl` and may add interaction columns such as `wx_wind_x_precip_scl`. In the assemble cell, **every** name in `numerical_cols` that **ends with `_scl`** is excluded from `StandardScaler` and concatenated after the scaled block (same rule for weather bases and `wx_*_scl`).
- **StandardScaler** (when `USE_STANDARD_SCALER`): train-fits mean and std on every column in `numerical_cols` that **does not** end with `_scl` — including binary flags, integer bins, counts, and target-encodings — then the `*_scl` slice is appended to build `features`.
- With `USE_PHASE3_COLUMN_LIST = True`, **only** names listed in `phase3_final_num_cols.json` that exist on the parquet are assembled, **in JSON order**.
- After the pipeline is built, train/val/test are **projected** to those numerics plus `DEP_DEL15` (and `class_weight` only if present on the parquet) before `cache()` and `fit` to reduce executor memory.


### Optional: Imputer on raw avg_Hourly

FE already median-fills weather. Set USE_ML_WEATHER_IMPUTER = True only if we need defensive ML imputation (e.g. new nulls).


In [0]:
available_cols = set(df_train_fe.columns)

# Raw avg_Hourly base names only (not *_scl). When using Phase 3 column list, restrict list to FINAL_NUM_COLS (legacy / diagnostics).
if USE_PHASE3_COLUMN_LIST and FINAL_NUM_COLS:
    _num_set = set(FINAL_NUM_COLS)
    weather_cols = sorted(
        c
        for c in df_train_fe.columns
        if c.startswith("avg_Hourly") and not c.endswith("_scl") and c in _num_set
    )
else:
    weather_cols = sorted(
        c
        for c in df_train_fe.columns
        if c.startswith("avg_Hourly") and not c.endswith("_scl")
    )

WX_SIX_BASES = [
    "avg_HourlyDryBulbTemperature",
    "avg_HourlyWindSpeed",
    "avg_HourlyRelativeHumidity",
    "avg_HourlySeaLevelPressure",
    "avg_HourlyVisibility",
    "avg_HourlyPrecipitation",
]
WX_SIX = [c for c in WX_SIX_BASES if c in available_cols]

ROLL_LOOKBACK_DAYS_IMPUTE = 7  # keep in sync with Phase 3.3 ROLL_LOOKBACK_DAYS_IMPUTE
CEILING_COL = "avg_feat_ceiling_hundreds_ft"
CEILING_SENTINEL = -1.0

_num_like = ("double", "int", "bigint", "float", "decimal")


def _extra_impute_cols(df):
    dt = {f.name: str(f.dataType).lower() for f in df.schema.fields}
    out = []
    for c in df.columns:
        t = dt.get(c, "")
        if not any(t.startswith(x) for x in _num_like):
            continue
        if c.startswith("avg_feat_") and "_category" not in c:
            out.append(c)
        elif c.startswith("last_WeatherType_"):
            out.append(c)
        elif c.startswith("last_feat_") and "_category" not in c:
            out.append(c)
        elif c.startswith("closest_station_") and c.endswith("_km"):
            out.append(c)
    return sorted(set(out))


def _apply_rolling_median_impute(df_train, df_val, df_test):
    """Phase 3.3 notebook logic: train daily medians, shifted rolling median, frozen val/test."""
    wx_cols = [c for c in df_train.columns if c.startswith("avg_Hourly")]
    impute_cols = sorted(set(wx_cols + _extra_impute_cols(df_train)))
    impute_cols = [c for c in impute_cols if c in df_train.columns]

    c_precip = "avg_HourlyPrecipitation"
    if c_precip in df_train.columns:
        def _fix_precip(dfx):
            return dfx.withColumn(
                c_precip,
                F.when(F.col(c_precip).cast("string") == "T", F.lit(0.0)).otherwise(
                    F.col(c_precip).cast("double")
                ),
            )

        df_train = _fix_precip(df_train)
        df_val = _fix_precip(df_val)
        df_test = _fix_precip(df_test)

    static_medians = {}
    for c in impute_cols:
        m = df_train.select(F.percentile_approx(c, 0.5).alias("m")).first()["m"]
        if m is None:
            m = CEILING_SENTINEL if c == CEILING_COL else 0.0
        static_medians[c] = float(m)

    _agg_exprs = [F.percentile_approx(c, 0.5).alias(c) for c in impute_cols]
    train_daily = df_train.groupBy("fl_date_d").agg(*_agg_exprs).orderBy("fl_date_d")
    pdf_daily = train_daily.toPandas().sort_values("fl_date_d")
    pdf_daily["fl_date_d"] = pd.to_datetime(pdf_daily["fl_date_d"])

    _roll_cols = []
    for c in impute_cols:
        s = pd.to_numeric(pdf_daily[c], errors="coerce")
        rcol = f"_roll_impute_{c}"
        pdf_daily[rcol] = s.shift(1).rolling(ROLL_LOOKBACK_DAYS_IMPUTE, min_periods=1).median()
        pdf_daily[rcol] = pdf_daily[rcol].fillna(static_medians[c])
        _roll_cols.append(rcol)

    _lookup_pdf = pdf_daily[["fl_date_d"] + _roll_cols].copy()
    lookup_sdf = spark.createDataFrame(_lookup_pdf)
    lookup_sdf = lookup_sdf.withColumn("fl_date_d", F.to_date("fl_date_d"))
    _joined = lookup_sdf.select(
        "fl_date_d",
        *[F.col(f"_roll_impute_{c}").alias(f"_fill_{c}") for c in impute_cols],
    )
    df_train = df_train.join(_joined, "fl_date_d", "left")
    for c in impute_cols:
        df_train = df_train.withColumn(c, F.coalesce(F.col(c), F.col(f"_fill_{c}"))).drop(
            f"_fill_{c}"
        )

    train_max_date = df_train.select(F.max("fl_date_d")).first()[0]
    tm = pd.to_datetime(train_max_date)
    _frozen_row = pdf_daily.loc[pdf_daily["fl_date_d"] == tm]
    if _frozen_row.empty:
        _frozen_row = pdf_daily.iloc[[-1]]
    frozen_impute = {
        c: float(_frozen_row[f"_roll_impute_{c}"].iloc[0]) for c in impute_cols
    }

    def _apply_frozen_impute(dfx):
        out = dfx
        for c, fv in frozen_impute.items():
            out = out.withColumn(c, F.coalesce(F.col(c), F.lit(fv)))
        return out

    df_val = _apply_frozen_impute(df_val)
    df_test = _apply_frozen_impute(df_test)
    return df_train, df_val, df_test, len(impute_cols)


if USE_ML_WEATHER_IMPUTER:
    df_train_fe, df_val_fe, df_test_fe, _n_impute = _apply_rolling_median_impute(
        df_train_fe, df_val_fe, df_test_fe
    )
    df_train_fe = df_train_fe.cache()
    df_val_fe = df_val_fe.cache()
    df_test_fe = df_test_fe.cache()
    print(
        f"Rolling median imputation: {ROLL_LOOKBACK_DAYS_IMPUTE}d lookback, {_n_impute} columns (Phase 3.3-style)."
    )
else:
    print("Skipping rolling median imputation (USE_ML_WEATHER_IMPUTER is False).")


Skipping rolling median imputation (USE_ML_WEATHER_IMPUTER is False).


### Assemble and scale

With `USE_PHASE3_COLUMN_LIST = True`, `numerical_cols` is built from `FINAL_NUM_COLS` (JSON order), intersected with parquet columns; `TARGET` and `class_weight` are excluded. With `False`, the legacy flight/time/target-encoding/weather lists are used.

The print line `numerical_cols from phase3_final_num_cols.json: …` gives the **feature dimension** for ML.

This cell then **projects** train/val/test to `numerical_cols` + `DEP_DEL15` (+ optional `class_weight`), **caches** the narrow frames, and optionally **repartitions** train before the fit cell.


In [0]:
available_cols = set(df_train_fe.columns)

if USE_PHASE3_COLUMN_LIST and FINAL_NUM_COLS is not None:
    _skip = {TARGET, "class_weight"}
    numerical_cols = [c for c in FINAL_NUM_COLS if c in available_cols and c not in _skip]
    if not numerical_cols:
        raise ValueError(
            "numerical_cols is empty: check phase3_final_num_cols.json vs parquet columns."
        )
    _missing_fe = [c for c in FINAL_NUM_COLS if c not in available_cols and c not in _skip]
    if _missing_fe:
        _show = _missing_fe[:25]
        print(
            f"Note: {len(_missing_fe)} name(s) in FINAL_NUM_COLS not in parquet (skipped): {_show}{'...' if len(_missing_fe) > 25 else ''}"
        )
else:
    flight_info = [
        "QUARTER",
        "MONTH",
        "DAY_OF_MONTH",
        "YEAR",
        "DISTANCE",
        "CRS_DEP_TIME",
        "dep_hour",
    ]
    time_features = [
        "is_morning_peak",
        "is_evening_peak",
        "is_weekend",
        "is_late_night",
        "dep_hour_sin",
        "dep_hour_cos",
        "day_sin",
        "day_cos",
        "month_sin",
        "month_cos",
        "quarter_sin",
        "quarter_cos",
    ]
    target_encoding = [
        "carrier_delay_rate",
        "origin_delay_rate",
        "dest_delay_rate",
        "route_delay_rate",
        "hour_delay_rate",
    ]
    weather_bins = ["precip_bin", "wind_bin", "vis_bin"]

    weather_feature_cols = []
    for base in WX_SIX:
        scl = f"{base}_scl"
        if scl in available_cols:
            weather_feature_cols.append(scl)
        else:
            weather_feature_cols.append(base)

    for base in weather_cols:
        if base in WX_SIX:
            continue
        weather_feature_cols.append(base)

    numerical_cols = []
    for col in flight_info + time_features + target_encoding + weather_bins:
        if col in available_cols:
            numerical_cols.append(col)
    numerical_cols.extend(weather_feature_cols)

cols_fe_scl = [c for c in numerical_cols if c.endswith("_scl")]
cols_to_scale = [c for c in numerical_cols if c not in cols_fe_scl]

if USE_PHASE3_COLUMN_LIST and FINAL_NUM_COLS is not None:
    print(f"numerical_cols from phase3_final_num_cols.json: {len(numerical_cols)} (present in parquet)")
else:
    print(f"WX_SIX bases in data: {WX_SIX}")
print(f"*_scl columns in vector: {cols_fe_scl}")
print(f"Total numeric inputs: {len(numerical_cols)} (scale {len(cols_to_scale)}, pass-through {len(cols_fe_scl)} *_scl)")

stages = []

if USE_STANDARD_SCALER and cols_to_scale:
    assembler_scale = VectorAssembler(
        inputCols=cols_to_scale,
        outputCol="numeric_features",
    )
    scaler = StandardScaler(
        inputCol="numeric_features",
        outputCol="numeric_features_scaled",
        withMean=True,
        withStd=True,
    )
    if cols_fe_scl:
        final_assembler = VectorAssembler(
            inputCols=["numeric_features_scaled"] + cols_fe_scl,
            outputCol="features",
        )
        stages.extend([assembler_scale, scaler, final_assembler])
    else:
        passthrough = VectorAssembler(
            inputCols=["numeric_features_scaled"],
            outputCol="features",
        )
        stages.extend([assembler_scale, scaler, passthrough])
else:
    final_only = VectorAssembler(
        inputCols=numerical_cols,
        outputCol="features",
    )
    stages.append(final_only)

pipeline = Pipeline(stages=stages)
print(f"Pipeline created with {len(stages)} stage(s):")
for i, st in enumerate(stages):
    print(f"  {i + 1}. {type(st).__name__}")

# Narrow to ML inputs only (preserves every name in numerical_cols; drops unused parquet columns)
for _dfn, _old in (("train", df_train_fe), ("val", df_val_fe), ("test", df_test_fe)):
    try:
        _old.unpersist(blocking=False)
    except Exception:
        pass

# Compute temporal sort key before narrow select drops date columns.
_date_src = None
if "fl_date_d" in df_train_fe.columns:
    _date_src = "fl_date_d"
elif all(c in df_train_fe.columns for c in ["YEAR", "MONTH", "DAY_OF_MONTH"]):
    _date_src = "ymd"
if _date_src == "fl_date_d":
    df_train_fe = df_train_fe.withColumn("fl_date_sort", F.date_format("fl_date_d", "yyyyMMdd").cast("int"))
    df_val_fe   = df_val_fe.withColumn(  "fl_date_sort", F.date_format("fl_date_d", "yyyyMMdd").cast("int"))
    df_test_fe  = df_test_fe.withColumn( "fl_date_sort", F.date_format("fl_date_d", "yyyyMMdd").cast("int"))
    print("fl_date_sort added from fl_date_d.")
elif _date_src == "ymd":
    df_train_fe = df_train_fe.withColumn("fl_date_sort", (F.col("YEAR")*10000 + F.col("MONTH")*100 + F.col("DAY_OF_MONTH")).cast("int"))
    df_val_fe   = df_val_fe.withColumn(  "fl_date_sort", (F.col("YEAR")*10000 + F.col("MONTH")*100 + F.col("DAY_OF_MONTH")).cast("int"))
    df_test_fe  = df_test_fe.withColumn( "fl_date_sort", (F.col("YEAR")*10000 + F.col("MONTH")*100 + F.col("DAY_OF_MONTH")).cast("int"))
    print("fl_date_sort added from YEAR/MONTH/DAY_OF_MONTH.")
else:
    print("WARNING: No date columns found — fl_date_sort not added. Time-series CV will use row order.")

_meta_cols = [TARGET]
if "class_weight" in df_train_fe.columns:
    _meta_cols.append("class_weight")
if _date_src is not None:
    _meta_cols.append("fl_date_sort")
pipeline_input_cols = list(dict.fromkeys(list(numerical_cols) + _meta_cols))
_missing_pi = [c for c in pipeline_input_cols if c not in df_train_fe.columns]
if _missing_pi:
    raise ValueError(f"pipeline_input_cols missing on train: {_missing_pi[:30]}")

df_train_fe = df_train_fe.select(*pipeline_input_cols)
df_val_fe = df_val_fe.select(*pipeline_input_cols)
df_test_fe = df_test_fe.select(*pipeline_input_cols)
_suffix = TARGET + (" + class_weight" if "class_weight" in pipeline_input_cols else "") + (" + fl_date_sort" if _date_src else "")
print(
    f"Projected to {len(pipeline_input_cols)} columns for fit/transform "
    f"({len(numerical_cols)} numeric features + {_suffix})."
)

if VERIFY_ROW_COUNTS:
    _PIPELINE_ROW_COUNTS = (
        df_train_fe.count(),
        df_val_fe.count(),
        df_test_fe.count(),
    )
    print(
        f"Narrow row counts — Train: {_PIPELINE_ROW_COUNTS[0]:,}, "
        f"Val: {_PIPELINE_ROW_COUNTS[1]:,}, Test: {_PIPELINE_ROW_COUNTS[2]:,}"
    )
else:
    _PIPELINE_ROW_COUNTS = None

df_train_fe = df_train_fe.cache()
df_val_fe = df_val_fe.cache()
df_test_fe = df_test_fe.cache()

if TRAIN_REPARTITION_BEFORE_FIT is not None and int(TRAIN_REPARTITION_BEFORE_FIT) > 0:
    _nr = int(TRAIN_REPARTITION_BEFORE_FIT)
    df_train_fe = df_train_fe.repartition(_nr)
    print(f"Train repartitioned to {_nr} partitions before fit (extra shuffle).")


numerical_cols from phase3_final_num_cols.json: 103 (present in parquet)
*_scl columns in vector: ['wx_precip_bin_x_wind_scl', 'wx_vis_bin_x_rh_scl', 'wx_morning_peak_x_vis_scl', 'wx_temp_scl_x_wind_scl', 'wind_bin_x_pressure_scl', 'precip_bin_x_temp_scl', 'holiday_x_precip_scl', 'weekend_x_rh_scl', 'late_night_x_precip_scl', 'late_night_x_wind_scl', 'prev2h_busy_x_precip_scl', 'prev2h_busy_x_wind_scl']
Total numeric inputs: 103 (scale 91, pass-through 12 *_scl)
Pipeline created with 3 stage(s):
  1. VectorAssembler
  2. StandardScaler
  3. VectorAssembler
fl_date_sort added from fl_date_d.
Projected to 105 columns for fit/transform (103 numeric features + DEP_DEL15 + fl_date_sort).
Narrow row counts — Train: 14,515,569, Val: 6,329,538, Test: 6,241,008


## Fit the pipeline on training data

Run the **assemble** cell first so `pipeline` exists and train/val/test are **narrow-cached**. If this cell fails, downstream cells will error until you re-run from the config cell.


In [0]:
# Fit learns (depending on flags):
# - Imputation medians (if USE_ML_WEATHER_IMPUTER)
# - StandardScaler mean & std on columns not ending with _scl (if USE_STANDARD_SCALER)

# Only fit on training data to avoid data leakage

print("Fitting pipeline on training data...")
print("(This may take a few minutes)\n")

_mlflow_was_disabled = False
if DISABLE_MLFLOW_AUTOLOG_FOR_FIT:
    try:
        import mlflow

        mlflow.spark.autolog(disable=True)
        _mlflow_was_disabled = True
    except Exception:
        pass

try:
    pipeline_model = pipeline.fit(df_train_fe)
finally:
    if _mlflow_was_disabled:
        try:
            import mlflow

            mlflow.spark.autolog(disable=False)
        except Exception:
            pass

print("\nPipeline fitted successfully")


Fitting pipeline on training data...
(This may take a few minutes)



2026/04/14 13:52:55 WARNING mlflow.utils.autologging_utils: MLflow spark autologging is known to be compatible with 3.2.1 <= pyspark <= 3.5.5, but the installed version is 4.0.0. If you encounter errors during autologging, try upgrading / downgrading pyspark to a compatible version, or try upgrading MLflow.
2026/04/14 13:52:55 WARNING mlflow.spark: With Pyspark >= 3.2, PYSPARK_PIN_THREAD environment variable must be set to false for Spark datasource autologging to work.



Pipeline fitted successfully


## Transform all three splits

In [0]:
# Apply the fitted pipeline to train, val, and test
# They all use the statistics learned from training data

print("Transforming splits...")
df_train_transformed = pipeline_model.transform(df_train_fe)
df_val_transformed = pipeline_model.transform(df_val_fe)
df_test_transformed = pipeline_model.transform(df_test_fe)




Transforming splits...


## Verify the output

Undersampled Phase 3.3 has **no** `class_weight` column; use **`features`** + **`DEP_DEL15`** only in ML parquets. Omit **`weightCol`** in estimators unless you add weights elsewhere.


Below runs **post-transform checks** (row counts vs loaded parquets, `features` length vs `numerical_cols`, non-null labels).


In [0]:
# Features + label (optional class_weight if legacy parquet)
print("Sample of transformed data:")
_out_cols = ["features", TARGET]
if "class_weight" in df_train_transformed.columns:
    _out_cols.append("class_weight")
df_train_transformed.select(*_out_cols).show(3)


Sample of transformed data:
+--------------------+---------+
|            features|DEP_DEL15|
+--------------------+---------+
|[0.29208408277309...|      0.0|
|[-1.0837497077779...|      0.0|
|[-1.0837497077779...|      0.0|
+--------------------+---------+
only showing top 3 rows


In [0]:
# Post-transform validation: row counts, vector dimension, labels (run after transform + sample preview)
if VERIFY_ROW_COUNTS and _PIPELINE_ROW_COUNTS is not None:
    _train_n, _val_n, _test_n = _PIPELINE_ROW_COUNTS
    assert df_train_transformed.count() == _train_n, "train row count mismatch"
    assert df_val_transformed.count() == _val_n, "val row count mismatch"
    assert df_test_transformed.count() == _test_n, "test row count mismatch"
else:
    print("Skipping row-count equality checks (VERIFY_ROW_COUNTS=False or counts not stored).")

_dim = df_train_transformed.select("features").first()[0].size
assert df_val_transformed.select("features").first()[0].size == _dim
assert df_test_transformed.select("features").first()[0].size == _dim
assert _dim == len(numerical_cols), f"features.size {_dim} != len(numerical_cols) {len(numerical_cols)}"

for split_name, dfx in [
    ("train", df_train_transformed),
    ("val", df_val_transformed),
    ("test", df_test_transformed),
]:
    n_null = dfx.filter(F.col(TARGET).isNull()).count()
    assert n_null == 0, f"{split_name}: {n_null} null {TARGET} labels"

print(
    f"Post-transform validation OK: features dim {_dim} == len(numerical_cols); "
    f"row checks={'on' if VERIFY_ROW_COUNTS and _PIPELINE_ROW_COUNTS is not None else 'skipped'}."
)


Post-transform validation OK: features dim 103 == len(numerical_cols); row checks=on.


## Save the pipeline model

In [0]:
# Save the fitted pipeline (optional — set WRITE_PIPELINE_MODEL = True in the config cell)
if WRITE_PIPELINE_MODEL:
    print(f"Saving pipeline to {PIPELINE_PATH}")
    pipeline_model.write().overwrite().save(PIPELINE_PATH)
    print("Saved successfully")
else:
    print("Skipping pipeline save (WRITE_PIPELINE_MODEL is False).")

Saving pipeline to dbfs:/student-groups/Group_01_02/phase3_pipeline_model_standard_undersample
Saved successfully


## Save ML-ready data splits

When `WRITE_ML_PARQUETS = True`, writes `features` and `DEP_DEL15` (and `class_weight` only if present) to paths built from **`ML_PARQUET_PREFIX`** (e.g. `phase2_df_undersample_train_ml.parquet`).

**Downstream training:** Load these paths in Phase 2.5; do **not** set `weightCol` unless the parquet includes `class_weight`.


In [0]:
# ML-ready frames: features + label (+ class_weight only if column exists)
if WRITE_ML_PARQUETS:
    _ml_sel = ["features", TARGET]
    if "class_weight" in df_train_transformed.columns:
        _ml_sel.append("class_weight")
    if "fl_date_sort" in df_train_transformed.columns:
        _ml_sel.append("fl_date_sort")
    df_train_ml = df_train_transformed.select(*_ml_sel)
    df_val_ml   = df_val_transformed.select(*_ml_sel)
    df_test_ml  = df_test_transformed.select(*_ml_sel)
    print(f"Saving ML-ready data ({ML_PARQUET_PREFIX}_train/val/test_ml.parquet)...")
    print("Schema:", df_train_ml.schema.simpleString())
    df_train_ml.write.mode("overwrite").parquet(ML_TRAIN_ML_PARQUET)
    df_val_ml.write.mode("overwrite").parquet(ML_VAL_ML_PARQUET)
    df_test_ml.write.mode("overwrite").parquet(ML_TEST_ML_PARQUET)
    print("Done")
    
else:
    print("Skipping ML parquet export (WRITE_ML_PARQUETS is False).")


Saving ML-ready data (phase2_df_undersample_train/val/test_ml.parquet)...
Schema: struct<features:vector,DEP_DEL15:double,fl_date_sort:int>
Done
